# Chat History API Reference


# `BaseChatMessageHistory: ABC`

Abstract base class for storing chat message history.

Concrete subclasses must expose `messages`, implement `clear`, and implement at least one of `add_message` or `add_messages` to support message addition.

## Fields

```python
messages: list[BaseMessage] # Messages returned by the history implementation
```

Retrieving this field or property may involve I/O and incur latency.

## Required subclass hooks

### `clear`

Removes all messages from the store.

```python
@abstractmethod
clear(
    self,
) -> None
```

## Optional subclass hooks

`add_messages` can be overridden for efficient bulk persistence. `aget_messages`, `aadd_messages`, and `aclear` can be overridden to provide native asynchronous implementations instead of the default executor wrappers.

## Methods

### `aget_messages`

Returns the stored messages asynchronously by reading `self.messages` through `run_in_executor`.

```python
async aget_messages(
    self,
) -> list[BaseMessage] # Stored messages
```

### `add_user_message`

Adds a human message, constructing a `HumanMessage` when given a string, and delegates to `add_message`.

This convenience method may be deprecated in a future release. Prefer `add_messages` to reduce persistence-layer round trips.

```python
add_user_message(
    self,
    message: HumanMessage | str, # Human message or string content to add
) -> None
```

### `add_ai_message`

Adds an AI message, constructing an `AIMessage` when given a string, and delegates to `add_message`.

This convenience method may be deprecated in a future release. Prefer `add_messages` to reduce persistence-layer round trips.

```python
add_ai_message(
    self,
    message: AIMessage | str, # AI message or string content to add
) -> None
```

### `add_message`

Adds one message. When the subclass overrides `add_messages`, the default implementation delegates using a one-element list.

```python
add_message(
    self,
    message: BaseMessage, # Message to store
) -> None
```

Raises `NotImplementedError` when the subclass implements neither `add_message` nor an overridden `add_messages` method.

### `add_messages`

Adds messages sequentially by calling `add_message` for each message. Subclasses can override this method for efficient bulk persistence.

```python
add_messages(
    self,
    messages: Sequence[BaseMessage], # Messages to store
) -> None
```

### `aadd_messages`

Asynchronously delegates bulk addition to `add_messages` through `run_in_executor`.

```python
async aadd_messages(
    self,
    messages: Sequence[BaseMessage], # Messages to store
) -> None
```

### `aclear`

Asynchronously delegates clearing to `clear` through `run_in_executor`.

```python
async aclear(
    self,
) -> None
```


# `InMemoryChatMessageHistory: BaseChatMessageHistory, BaseModel`

Stores chat messages in an in-process list.

## Fields

```python
messages: list[BaseMessage] = Field(default_factory=list) # Messages stored in memory
```

## Constructor

```python
InMemoryChatMessageHistory(
    *,
    messages: list[BaseMessage] = Field(default_factory=list), # Initial stored messages
) -> None
```

## Methods

### `aget_messages`

Returns the current `messages` list directly without using an executor or creating a copy.

```python
async aget_messages(
    self,
) -> list[BaseMessage] # Live stored-message list
```

### `add_message`

Appends one message to the in-memory list.

```python
add_message(
    self,
    message: BaseMessage, # Message to append
) -> None
```

### `aadd_messages`

Calls the inherited synchronous `add_messages` implementation directly. Messages are appended in input order through `add_message`.

```python
async aadd_messages(
    self,
    messages: Sequence[BaseMessage], # Messages to append
) -> None
```

### `clear`

Replaces the current message list with a new empty list.

```python
clear(
    self,
) -> None
```

### `aclear`

Calls `clear` directly without using an executor.

```python
async aclear(
    self,
) -> None
```

In [ ]:
from collections.abc import Sequence # Import the sequence type for multiple messages
from pydantic import BaseModel, Field # Import Pydantic model and field utilities
from langchain_core.messages import BaseMessage # Import the base message type
from langchain_core.chat_history import BaseChatMessageHistory # Import the base chat-history class

class InMemoryChatMessageHistory(BaseChatMessageHistory, BaseModel): # Define an in-memory chat-history implementation
    messages: list[BaseMessage] = Field(default_factory=list) # Store messages in a newly created list

    async def aget_messages(self) -> list[BaseMessage]: # Retrieve the stored messages asynchronously
        return self.messages # Return the live message list without creating a copy

    def add_message(self, message: BaseMessage) -> None: # Add one message to the history
        self.messages.append(message) # Append the message to the in-memory list

    async def aadd_messages(self, messages: Sequence[BaseMessage]) -> None: # Add multiple messages asynchronously
        self.add_messages(messages) # Call the inherited synchronous bulk-addition method directly

    def clear(self) -> None: # Remove all stored messages
        self.messages = [] # Replace the current list with a new empty list

    async def aclear(self) -> None: # Remove all stored messages asynchronously
        self.clear() # Call the synchronous clear method directly